In [2]:
import sys
import os
import json
import urllib3
import requests
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)


Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [22]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-04-23T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
group_id = 6755399443003054

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = f'ownerName EQ "{owner}" and lastModified >= "{start}"'

        # Set up the request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/groups/{group_id}?tql={tql}&fields=associatedGroups&resultStart=0&resultLimit=10000')

        # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Ensure 'data' is a list for each result
    normalized_results = []
    for result in final_results:
        if isinstance(result.get('data'), dict):
            normalized_results.append(result['data'])
        elif isinstance(result.get('data'), list):
            normalized_results.extend(result['data'])

    if normalized_results:
        observed_src = pd.json_normalize(normalized_results)
        print(f"\nRetrieved {len(observed_src)} observed indicators.")
    else:
        print("\nNo valid data to normalize.")
else:
    print("\nNo indicators retrieved.")



Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 3 observed indicators.


In [23]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,name,upVoteCount,downVoteCount,firstSeen,lastModified,legacyLink,createdBy.id,createdBy.userName,createdBy.firstName,createdBy.lastName,createdBy.pseudonym,createdBy.owner
0,6755399443003054,2025-03-28T13:49:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/groups...,Campaign,HTOC Observation Gap Testing Group,0,0,2025-03-28T09:47:00Z,2025-04-24T12:15:05Z,https://hvs.threatconnect.com/auth/campaign/ca...,671,wesley.smith@hhs.gov,Wesley,Smith /HHS-HTOC/,wes-htoc,HTOC Org
1,6755399443003054,2025-03-28T13:49:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/groups...,Campaign,HTOC Observation Gap Testing Group,0,0,2025-03-28T09:47:00Z,2025-04-24T12:15:05Z,https://hvs.threatconnect.com/auth/campaign/ca...,671,wesley.smith@hhs.gov,Wesley,Smith /HHS-HTOC/,wes-htoc,HTOC Org
2,6755399443003054,2025-03-28T13:49:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/groups...,Campaign,HTOC Observation Gap Testing Group,0,0,2025-03-28T09:47:00Z,2025-04-24T12:15:05Z,https://hvs.threatconnect.com/auth/campaign/ca...,671,wesley.smith@hhs.gov,Wesley,Smith /HHS-HTOC/,wes-htoc,HTOC Org


In [24]:
final_results

[{'data': {'id': 6755399443003054,
   'dateAdded': '2025-03-28T13:49:15Z',
   'ownerId': 9,
   'ownerName': 'HTOC Org',
   'webLink': 'https://hvs.threatconnect.com/#/details/groups/6755399443003054',
   'type': 'Campaign',
   'name': 'HTOC Observation Gap Testing Group',
   'createdBy': {'id': 671,
    'userName': 'wesley.smith@hhs.gov',
    'firstName': 'Wesley',
    'lastName': 'Smith /HHS-HTOC/',
    'pseudonym': 'wes-htoc',
    'owner': 'HTOC Org'},
   'upVoteCount': '0',
   'downVoteCount': '0',
   'associatedGroups': {},
   'firstSeen': '2025-03-28T09:47:00Z',
   'lastModified': '2025-04-24T12:15:05Z',
   'legacyLink': 'https://hvs.threatconnect.com/auth/campaign/campaign.xhtml?campaign=6755399443003054'},
  'status': 'Success'},
 {'data': {'id': 6755399443003054,
   'dateAdded': '2025-03-28T13:49:15Z',
   'ownerId': 9,
   'ownerName': 'HTOC Org',
   'webLink': 'https://hvs.threatconnect.com/#/details/groups/6755399443003054',
   'type': 'Campaign',
   'name': 'HTOC Observation 